In [12]:
import json
import re

In [27]:
with open(r'./data\protokolle\20_012_2022-01-14.json') as f:
    d = json.load(f)
    text = d['text']
    text = re.sub('BÜNDNIS\s*90\/DIE\s*GRÜNEN', 'GRÜNE', text)

In [28]:
name_match = r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?|\w+(?:\s\w+)*)'
partei_match = r'(CDU\/CSU|GRÜNE|FDP|AfD|SPD)' # Parteilos + alte Parteien fehlen!
kommentar_match = rf'{name_match}\s+\[{partei_match}\]: (.*?)[–\)]'
beifall_match = r'[(?:–\s)\(]Beifall .*?[–\)]'
#beifall_match1 = r'\(Beifall .*?[–\)]'
zuruf_match = r'[(?:–\s)\(]Zuruf .*?[–\)]'
speaker_match = rf'{name_match}\s\({partei_match}\):|\n\n\n{name_match}, (.*?):'


In [29]:
special_speaker_match = rf'\n\n\n{name_match}, (.*?):'
speeches_raw = re.split(speaker_match, text)[1:]
speeches_raw

[None,
 None,
 'Christian Lindner',
 'Bundesminister der Finanzen',
 '\nSehr geehrter Herr Präsident! Meine Damen und Herren! Die Finanzpolitik wurde in den vergangenen Jahren durch die Coronapandemie bestimmt. Die Finanzpolitik wird auch gegenwärtig von der Pandemie geprägt. Die Aufgabe der kommenden vier Jahre wird es sein, aus dem finanzpolitischen Krisenmodus in den finanzpolitischen Gestaltungsmodus zu wechseln.\n(Beifall bei der FDP und dem GRÜNE sowie bei Abgeordneten der SPD)\nErstens. Die Bundesregierung arbeitet daran, im Jahr 2023 zum Regelfall der Schuldenbremse des Grundgesetzes zurückzukehren. In den Folgejahren ist es mein Ziel, die deutsche Schuldenquote zu reduzieren. Es ist ein Gebot der Klugheit, nach einer Krise die fiskalische Handlungsfähigkeit des Staates für künftige Krisen zu stärken.\n(Beifall bei der FDP und dem GRÜNE sowie bei Abgeordneten der SPD\xa0– Zuruf des Abg. Kay Gottschalk [AfD])\nNoch länger werden wir die Folgen der Pandemie sehen. Deshalb reservi

In [17]:
import xml.etree.ElementTree as ET

In [16]:
tree = ET.parse('./data/MDB_STAMMDATEN.XML')
root = tree.getroot()
bt_list = []
for mdb in root.findall('MDB'):
    dict = {
        'last_name':mdb.find('.//NACHNAME').text,
        'first_name':mdb.find('.//VORNAME').text,
        'anrede':mdb.find('.//ANREDE_TITEL').text,
        'party':mdb.find('.//PARTEI_KURZ').text,
        'election_period':mdb.find('.//WP').text
    }
    bt_list.append(dict)

bt_tuples = [(f"{adbt['first_name']} {adbt['last_name']}", adbt['party'], adbt['election_period']) for adbt in bt_list]
bt_tuples

[('Manfred Abelein', 'CDU', '5'),
 ('Ernst Achenbach', 'FDP', '3'),
 ('Annemarie Ackermann', 'CDU', '2'),
 ('Else Ackermann', 'CDU', '11'),
 ('Ulrich Adam', 'CDU', '12'),
 ('Rudolf Adams', 'SPD', '5'),
 ('Raban Adelmann', 'CDU', '3'),
 ('Konrad Adenauer', 'CDU', '1'),
 ('Brigitte Adler', 'SPD', '11'),
 ('Eduard Adorno', 'CDU', '4'),
 ('Jochen Aerssen', 'CDU', '8'),
 ('Willi Agatz', 'KPD', '1'),
 ('Conrad Ahlers', 'SPD', '7'),
 ('Adolf Ahrens', 'DP', '1'),
 ('Hermann Ahrens', 'SPD', '5'),
 ('Karl Ahrens', 'SPD', '6'),
 ('Heinrich Aigner', 'CSU', '3'),
 ('Siegbert Alber', 'CDU', '6'),
 ('Johannes Albers', 'CDU', '1'),
 ('Luise Albertz', 'SPD', '1'),
 ('Ina Albowitz', 'FDP', '12'),
 ('Ernst Albrecht', 'CDU', '2'),
 ('Lisa Albrecht', 'SPD', '1'),
 ('Michael Albrecht', 'CDU', '11'),
 ('Peter Alltschekow', 'SPD', '12'),
 ('Odal Alten-Nordheim', 'CDU', '6'),
 ('Walter Althammer', 'CSU', '4'),
 ('Walter Altherr', 'CDU', '12'),
 ('Jakob Altmaier', 'SPD', '1'),
 ('Wilhelm Altvater', 'SPD', '3'),

In [37]:
bt_list_json = json.dumps(bt_list, indent=4)
with open("bt_list.json", "w") as json_file:
    json_file.write(bt_list_json)

In [39]:
def find_party_by_name(name, tuple_list):
    for item in tuple_list:
        if name in item[0]:
            return item[1]
    return "Party not found"

In [41]:
#speaker_match = rf'{name_match}\s\({partei_match}\):'
#speaker_match = rf'{name_match}\s\({partei_match}\):|\n\n\n{name_match}, (.*?):'

speeches_raw = re.split(speaker_match, text)[1:]
speeches = []
for i in range(0, len(speeches_raw), 5):
    if speeches_raw[i] != None and speeches_raw[i+1] != None:
        name = speeches_raw[i].strip()    
        party = speeches_raw[i+1].strip()
    else:
        name = speeches_raw[i+2].strip()
        party = find_party_by_name(name, bt_tuples)
        if party == "Party not found":
            print(f'party not found for {name}')
    
    speech = {
        'redner':{
            'name':name,
            'party':party
        },
        'text': speeches_raw[i+4].strip()

    }
    speeches.append(speech)



In [42]:
print(speeches[0])

{'redner': {'name': 'Christian Lindner', 'party': 'FDP'}, 'text': 'Sehr geehrter Herr Präsident! Meine Damen und Herren! Die Finanzpolitik wurde in den vergangenen Jahren durch die Coronapandemie bestimmt. Die Finanzpolitik wird auch gegenwärtig von der Pandemie geprägt. Die Aufgabe der kommenden vier Jahre wird es sein, aus dem finanzpolitischen Krisenmodus in den finanzpolitischen Gestaltungsmodus zu wechseln.\n(Beifall bei der FDP und dem GRÜNE sowie bei Abgeordneten der SPD)\nErstens. Die Bundesregierung arbeitet daran, im Jahr 2023 zum Regelfall der Schuldenbremse des Grundgesetzes zurückzukehren. In den Folgejahren ist es mein Ziel, die deutsche Schuldenquote zu reduzieren. Es ist ein Gebot der Klugheit, nach einer Krise die fiskalische Handlungsfähigkeit des Staates für künftige Krisen zu stärken.\n(Beifall bei der FDP und dem GRÜNE sowie bei Abgeordneten der SPD\xa0– Zuruf des Abg. Kay Gottschalk [AfD])\nNoch länger werden wir die Folgen der Pandemie sehen. Deshalb reservieren 

In [48]:
speech_text =  speeches[5]['text']
for speech in speeches:
    comments = []
    for match in re.finditer(kommentar_match,speech['text']):
        comment = {
            'commentator': {
                'name': match.group(1),
                'party': match.group(2)
            },
            'text': match.group(3),
            'preceding_context': speech_text[:match.start()] 
        }
        comments.append(comment)
    speech['comments'] = comments
    #for match in re.finditer(beifall_match):
        
    #TODO Beifall und Zuruf


In [ ]:
json_string = json.dumps(speeches, indent=4) 

# Write JSON string to a file
with open("parsed_example.json", "w") as json_file:
    json_file.write(json_string)

# List of Abgeordnete

# For all Plenarprotokolle in a given folder

In [51]:
import os

In [61]:
data_path = './data/protokolle/'

<class 'list'>


In [63]:
for protokoll in os.listdir(data_path):
    with open(data_path+protokoll) as f:
        doc = json.load(f)
        text = doc['text']
    speaker_match = rf'{name_match}\s\({partei_match}\):'

    speeches_raw = re.split(speaker_match, text)[1:]
    speeches = []
    for i in range(0, len(speeches_raw), 3):
        speech = {  
            'redner':{
                'name':speeches_raw[i].strip(),
                'party':speeches_raw[i+1].strip()
            },
            #'text': speeches_raw[i+2].strip()
            'text': re.split(r'\n\n',speeches_raw[i+2])[0].strip() # cut off speakers like State ministers without party affiliation
        }

        comments = []
        for match in re.finditer(kommentar_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3),
                'preceding_context': speech_text[:match.start()] 
            }
            comments.append(comment)
        speech['comments'] = comments

        speeches.append(speech)

    json_string = json.dumps(speeches, indent=4) 
    path = f'./data/parsed_comments/{doc['wahlperiode']}_{doc['dokumentnummer'].split(r'/')[1].zfill(3)}_{doc['datum']}_parsed.json'
    # Write JSON string to a file
    with open(path, "w") as json_file:
        json_file.write(json_string)   



In [5]:
kommentare = re.findall(kommentar_match, text)
kommentare

[('Dr.\xa0Anja Reinalter',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Zum Thema Nachhaltigkeit!'),
 ('Dr.\xa0Anja Reinalter',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Es geht nicht um Landwirte! Es geht um die Nachhaltigkeitsziele!'),
 ('Dr.\xa0Jan-Niclas Gesenhues',
  'BÜNDNIS\xa090/DIE GRÜNEN',
  'Auf die Schiene! Genau!'),
 ('Andreas Bleck',
  'AfD',
  'Ihr habt Deutschland abgewirtschaftet! Ihr werdet bei der nächsten Bundestagswahl abgestraft!\xa0'),
 ('Albrecht Glaser', 'AfD', 'Dreckiger Schmutz!'),
 ('Andreas Bleck', 'AfD', 'Das macht ihr! Noch nie ging es uns so schlecht!'),
 ('Andreas Bleck', 'AfD', 'Das entscheiden immer noch die Wähler!'),
 ('Andreas Bleck', 'AfD', 'Vor allem!'),
 ('Alexander Dobrindt', 'CDU/CSU', 'So ist es!'),
 ('Frank Bsirske', 'BÜNDNIS\xa090/DIE GRÜNEN', 'Blödsinn!'),
 ('Jakob Blankenburg',
  'SPD',
  'Wer war denn die letzten Jahre Landwirtschaftsminister?'),
 ('Bettina Hagedorn', 'SPD', 'Bei Ihnen passt gar nichts zusammen!'),
 ('Dr.\xa0Jan-Niclas Gesenhues',
  'BÜNDNIS

In [3]:
import json

In [43]:
beifall_match = r'[(?:–\s)\(]Beifall .*?[–\)]'
#beifall_match1 = r'\(Beifall .*?[–\)]'

zuruf_match = r'[(?:–\s)\(]Zuruf .*?[–\)]'
re.findall(zuruf_match, text)

[' Zuruf des Abg. Kay Gottschalk [AfD])',
 '(Zuruf von der LINKEN: Schon wieder?)',
 '(Zuruf der Abg. Antje Tillmann [CDU/CSU])',
 '(Zuruf von der AfD: Moin!)',
 ' Zuruf von der AfD: Lächerliches Geplänkel!)',
 '(Zuruf von der AfD: Wir haben doch gar keine Kultur!)',
 '(Zuruf von der CDU/CSU: Wer hat denn damit angefangen?)',
 ' Zuruf der Abg. Renate Künast [GRÜNE])',
 '(Zuruf von der LINKEN)',
 ' Zuruf der Abg. Amira Mohamed Ali [DIE LINKE])',
 ' Zuruf der Abg. Amira Mohamed Ali [DIE LINKE])',
 '(Zuruf der Abg. Filiz Polat [GRÜNE])',
 '(Zuruf vom GRÜNE: Das steht doch alles im Koalitionsvertrag drin! Sie müssen das nur lesen!)',
 '(Zuruf des Abg. Dr.\xa0Matthias Miersch [SPD])',
 '(Zuruf vom GRÜNE)',
 ' Zuruf des Abg. Albert Stegemann [CDU/CSU])',
 '(Zuruf von der SPD: Das ist Bürokratendenken!)',
 '(Zuruf von der SPD: Quatsch! Das ist ja peinlich!)',
 '(Zuruf von der SPD: Das ist aber eine schlechte Bilanz!)',
 ' Zuruf des Abg. Florian Hahn [CDU/CSU])',
 ' Zuruf von der AfD: Ist doch

In [4]:
with open(r'./data\protokolle\20_012_2022-01-14.json') as f:
    d = json.load(f)
    text = d['text']

In [5]:
with open("./testtext2.txt" , 'w', encoding='utf8') as f:
    f.write(text)

In [ ]:
satzende_match = '(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'

# Sonderfälle

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml